# Detection de chutes : reentrainement (Transformer + keypoints COCO)
**Avant de lancer :** Runtime -> Change runtime type -> **T4 GPU**. Dossiers attendus sur Drive : `Memoir/Fall/Keypoints_CSV` et `Memoir/No_Fall/Keypoints_CSV`.

Execute les cellules dans l'ordre (Shift+Entree). Resultats : `best_model_v2.pth` et `config.json` dans `Memoir/` (et telecharges automatiquement).

## 1. Configuration

In [ ]:
import os, json, glob, random, time, pickle
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')     # une fenetre s'ouvre : choisis le MEME compte Google que ton Drive
    IN_COLAB = True
except ImportError:
    pass

# ============ A MODIFIER SI BESOIN ============
DRIVE_DIR  = '/content/drive/MyDrive/Memoir'
FALL_DIR   = f'{DRIVE_DIR}/Fall/Keypoints_CSV'
NOFALL_DIR = f'{DRIVE_DIR}/No_Fall/Keypoints_CSV'
MAX_PER_CLASS = 2000      # mets 6000 pour tout charger (plus long)
EPOCHS = 40
# Plus tard : donnees VR (meme format CSV que FallVision), ex :
# EXTRA_DIRS = [('/content/drive/MyDrive/Memoir/VR/Fall', 1), ('/content/drive/MyDrive/Memoir/VR/No_Fall', 0)]
EXTRA_DIRS = []
# ===============================================


def find_dir(preferred, parent_name):
    """Renvoie le dossier de CSV ; s'il n'existe pas, le cherche dans tout le Drive."""
    if os.path.isdir(preferred) and glob.glob(f'{preferred}/*.csv'):
        return preferred
    print('Introuvable :', preferred, '-> recherche automatique dans le Drive...')
    base = '/content/drive'
    for root, dirs, _ in os.walk(base, followlinks=True):
        if root[len(base):].count('/') >= 5:
            dirs[:] = []
            continue
        if (os.path.basename(root) == 'Keypoints_CSV'
                and os.path.basename(os.path.dirname(root)) == parent_name
                and glob.glob(f'{root}/*.csv')):
            print('  trouve :', root)
            return root
    raise FileNotFoundError(
        f"Aucun dossier '{parent_name}/Keypoints_CSV' avec des CSV dans ton Drive. Verifie que le dossier "
        "est dans 'Mon Drive' (pas seulement 'Partages avec moi') et que le bon compte Google est connecte.")


FALL_DIR = find_dir(FALL_DIR, 'Fall')
NOFALL_DIR = find_dir(NOFALL_DIR, 'No_Fall')

# Dossier de sortie (modele, config, cache) : a cote des donnees si on peut y ecrire
OUT_DIR = os.path.dirname(os.path.dirname(FALL_DIR))
try:
    os.makedirs(OUT_DIR, exist_ok=True)
    open(f'{OUT_DIR}/.w', 'w').close(); os.remove(f'{OUT_DIR}/.w')
except Exception:
    OUT_DIR = '/content/fall_out'
    os.makedirs(OUT_DIR, exist_ok=True)
    print('Dossier en lecture seule -> sorties dans', OUT_DIR, '(telechargees a la fin)')
print('Fall   :', FALL_DIR, '\nNo_Fall:', NOFALL_DIR, '\nSorties:', OUT_DIR)

## 2. Code commun (a ne pas modifier : identique a `webcam.py`)

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# 17 keypoints COCO, dans l'ordre exact utilise par le modele
KP_NAMES = ['Nose', 'Left Eye', 'Right Eye', 'Left Ear', 'Right Ear',
            'Left Shoulder', 'Right Shoulder', 'Left Elbow', 'Right Elbow',
            'Left Wrist', 'Right Wrist', 'Left Hip', 'Right Hip',
            'Left Knee', 'Right Knee', 'Left Ankle', 'Right Ankle']
FLIP_PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]

T = 120               # nombre de frames par sequence
FPS_ASSUMED = 30      # le dataset est a 30 fps -> 120 frames = 4 secondes
WINDOW_SEC = T / FPS_ASSUMED


def normalize_seq(p):
    """p : (T, 17, 2) en coordonnees IMAGE (y vers le bas) -> (T, 34).
    Centre sur les hanches de la 1ere frame, divise par la taille du torse."""
    p = np.asarray(p, dtype=np.float32)
    hips = p[:, 11:13, :].mean(1)
    shoulders = p[:, 5:7, :].mean(1)
    torso = np.linalg.norm(shoulders - hips, axis=1)
    scale = max(float(np.median(torso)), 1.0)
    out = (p - hips[0]) / scale
    return np.clip(out, -10, 10).reshape(len(p), -1).astype(np.float32)


class FallTransformer(nn.Module):
    def __init__(self, input_dim=34, num_heads=2, num_layers=2, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Linear(input_dim, hidden_dim)
        layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads,
                                           dim_feedforward=128, dropout=0.1,
                                           batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        # encodage de position appris : indique au modele l'ORDRE des frames
        self.pos = nn.Parameter(torch.randn(1, T, hidden_dim) * 0.02)
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(),
                                        nn.Dropout(0.3), nn.Linear(32, 2))

    def forward(self, x):
        x = self.embedding(x) + self.pos[:, :x.size(1)]
        x = self.transformer(x)
        return self.classifier(x.mean(dim=1))

## 3. Chargement des donnees + diagnostics (axe Y, longueur des videos)

In [ ]:
IDX = {n: i for i, n in enumerate(KP_NAMES)}
MAX_LEN = 600   # on garde au maximum 600 frames par video


def read_clip(path):
    """CSV (Frame, Keypoint, X, Y, Confidence) -> tableau (frames, 17, 2)."""
    df = pd.read_csv(path, usecols=['Frame', 'Keypoint', 'X', 'Y'])
    df = df[df.groupby('Frame')['Keypoint'].transform('size') == 17]   # frames completes seulement
    if df.empty:
        return None
    k = df['Keypoint'].map(IDX)
    if k.isna().any():   # noms inattendus -> on suit l'ordre des lignes
        df = df.sort_values('Frame', kind='stable')
    else:
        df = df.assign(k=k.astype(int)).sort_values(['Frame', 'k'])
    arr = df[['X', 'Y']].to_numpy(np.float32)
    P = np.nan_to_num(arr.reshape(-1, 17, 2))
    return P[:MAX_LEN]


def motion_peak(P):
    """Index de la frame ou le mouvement est le plus fort (la chute dans une video Fall)."""
    hips, sh = P[:, 11:13].mean(1), P[:, 5:7].mean(1)
    scale = max(float(np.median(np.linalg.norm(sh - hips, axis=1))), 1.0)
    d = np.linalg.norm(np.diff(P, axis=0), axis=2).mean(1) / scale
    if len(d) == 0:
        return 0
    k = min(15, len(d))
    return int(np.argmax(np.convolve(d, np.ones(k) / k, mode='same')))


def load_class(folder, label, max_n):
    files = sorted(glob.glob(f'{folder}/*.csv'))
    random.Random(0).shuffle(files)            # melange : on ne prend pas toujours les memes sessions
    files = files[:max_n]

    def work(f):
        try:
            return f, read_clip(f)
        except Exception:
            return f, None

    out = []
    with ThreadPoolExecutor(8) as ex:
        for f, P in tqdm(ex.map(work, files), total=len(files), desc=f'classe {label}'):
            if P is not None and len(P) >= 30:
                out.append((os.path.basename(f), P, label))
    return out


CACHE = f'{OUT_DIR}/clips_cache.pkl'
settings = dict(f=FALL_DIR, n=NOFALL_DIR, m=MAX_PER_CLASS, x=EXTRA_DIRS)
clips = None
if os.path.exists(CACHE):
    with open(CACHE, 'rb') as fh:
        cached = pickle.load(fh)
    if cached.get('settings') == settings:
        clips = cached['clips']
        print('Cache charge :', len(clips), 'clips')
if clips is None:
    clips = load_class(FALL_DIR, 1, MAX_PER_CLASS) + load_class(NOFALL_DIR, 0, MAX_PER_CLASS)
    for d, lab in EXTRA_DIRS:
        clips += load_class(d, lab, MAX_PER_CLASS)
    with open(CACHE, 'wb') as fh:
        pickle.dump(dict(settings=settings, clips=clips), fh)

labels_all = np.array([c[2] for c in clips])
print('Fall:', int((labels_all == 1).sum()), '| No_Fall:', int((labels_all == 0).sum()))

# ---------- Diagnostic 1 : axe Y ----------
def head_minus_feet(P):
    return float(np.median(P[:10, 0, 1]) - np.median(P[:10, 15:17, 1]))

vals = [head_minus_feet(P) for _, P, l in clips if l == 0][:80]
Y_FLIP = bool(np.median(vals) > 0)     # tete "plus bas" que les pieds => axe Y vers le haut
print('Axe Y vers le haut (inverse automatiquement) :', Y_FLIP)
if Y_FLIP:
    clips = [(n, P * np.array([1, -1], dtype=np.float32), l) for n, P, l in clips]

# ---------- Diagnostic 2 : longueur des videos ----------
for lab, name in [(1, 'Fall'), (0, 'No_Fall')]:
    L = np.array([len(P) for _, P, l in clips if l == lab])
    print(f'{name:8s} frames/video : min {L.min()}  moyenne {L.mean():.0f}  max {L.max()}')
mf = np.mean([len(P) for _, P, l in clips if l == 1])
mn = np.mean([len(P) for _, P, l in clips if l == 0])
if not (0.8 < mf / mn < 1.25):
    print('ATTENTION : les longueurs Fall / No_Fall sont tres differentes. Le code utilise '
          'des fenetres de 120 frames pour eviter que le modele apprenne la longueur.')

## 4. Entrainement

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import f1_score

# ---------- Split train / validation (par sessions pour eviter la triche) ----------
names = [c[0] for c in clips]
y_all = np.array([c[2] for c in clips])
groups = np.array([n[:11] for n in names])           # ex: 20240912_10 (date + heure)
tr_idx = va_idx = None
if len(set(groups)) >= 20:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr_idx, va_idx = next(gss.split(np.zeros(len(clips)), y_all, groups))
    if len(set(y_all[va_idx])) < 2 or len(set(y_all[tr_idx])) < 2:
        tr_idx = None
if tr_idx is None:
    tr_idx, va_idx = train_test_split(np.arange(len(clips)), test_size=0.2,
                                      stratify=y_all, random_state=42)
    print('Split aleatoire stratifie')
else:
    print('Split par sessions')
print('Train :', len(tr_idx), '| Val :', len(va_idx),
      '| Fall en val :', int((y_all[va_idx] == 1).sum()))


def torso_scale(W):
    d = np.linalg.norm(W[:, 5:7].mean(1) - W[:, 11:13].mean(1), axis=1)
    return max(float(np.median(d)), 1.0)


class ClipDS(Dataset):
    def __init__(self, clips, idx, train):
        self.items = [(clips[i][1], clips[i][2], motion_peak(clips[i][1])) for i in idx]
        self.train = train
        self.rng = np.random.default_rng(None if train else 0)

    def __len__(self):
        return len(self.items)

    def window(self, P, peak, label):
        L = len(P)
        s = self.rng.uniform(0.75, 1.25) if self.train else 1.0     # vitesse variable (webcam != 30 fps)
        span = T * s
        if L <= span:
            start = 0.0
        elif label == 1:
            center = float(np.clip(peak - span / 2, 0, L - span))
            if self.train:
                lo, hi = max(0.0, peak - 0.75 * span), min(L - span, peak - 0.25 * span)
                start = self.rng.uniform(lo, hi) if hi > lo else center
            else:
                start = center
        else:
            start = self.rng.uniform(0, L - span) if self.train else (L - span) / 2
        t = np.clip(start + np.arange(T) * s, 0, L - 1)
        i0 = np.floor(t).astype(int)
        i1 = np.minimum(i0 + 1, L - 1)
        w = (t - i0)[:, None, None].astype(np.float32)
        return P[i0] * (1 - w) + P[i1] * w

    def augment(self, W):
        W = W.copy()
        if self.rng.random() < 0.5:                       # miroir gauche/droite
            W[..., 0] *= -1
            for a, b in FLIP_PAIRS:
                W[:, [a, b]] = W[:, [b, a]]
        ang = np.deg2rad(self.rng.uniform(-15, 15))       # camera un peu penchee
        c, s_ = np.cos(ang), np.sin(ang)
        W = W @ np.array([[c, -s_], [s_, c]], dtype=np.float32).T
        W = W + self.rng.normal(0, 0.01 * torso_scale(W), W.shape).astype(np.float32)
        return W

    def __getitem__(self, i):
        P, label, peak = self.items[i]
        W = self.window(P, peak, label)
        if self.train:
            W = self.augment(W)
        return torch.from_numpy(normalize_seq(W)), int(label)


train_ds, val_ds = ClipDS(clips, tr_idx, True), ClipDS(clips, va_idx, False)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

# ---------- Modele ----------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = FallTransformer().to(device)
cnt = np.bincount(y_all[tr_idx], minlength=2).astype(np.float32)
weights = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def run_val():
    model.eval()
    probs, labs, tot = [], [], 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            out = model(xb.to(device))
            tot += criterion(out, yb.to(device)).item() * len(yb)
            probs.append(torch.softmax(out, 1)[:, 1].cpu().numpy())
            labs.append(yb.numpy())
    return tot / len(val_ds), np.concatenate(probs), np.concatenate(labs)


best_f1, MODEL_PATH = -1.0, f'{OUT_DIR}/best_model_v2.pth'
print(f"{'epoch':>5} {'train_loss':>10} {'train_acc':>10} {'val_loss':>9} {'val_acc':>8} {'val_f1':>7}")
for epoch in range(1, EPOCHS + 1):
    model.train()
    tl, ok, n = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        tl += loss.item() * len(yb); ok += (out.argmax(1) == yb).sum().item(); n += len(yb)
    scheduler.step()
    vl, vp, vy = run_val()
    pred = (vp > 0.5).astype(int)
    f1 = f1_score(vy, pred, average='macro')
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), MODEL_PATH)
    if epoch == 1 or epoch % 5 == 0:
        print(f'{epoch:>5} {tl / n:>10.4f} {ok / n:>10.4f} {vl:>9.4f} {(pred == vy).mean():>8.4f} {f1:>7.4f}')
print('Meilleur F1 (validation) :', round(best_f1, 4), '->', MODEL_PATH)

## 5. Evaluation, seuil, sauvegarde

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
_, vp, vy = run_val()
print(classification_report(vy, (vp > 0.5).astype(int), target_names=['No Fall', 'Fall'], digits=3))

# Seuil conseille : meilleur F1 sur la classe Fall avec un rappel >= 90 %
best_thr, best_score = 0.5, -1.0
for thr in np.arange(0.30, 0.96, 0.05):
    p = (vp > thr).astype(int)
    tp = ((p == 1) & (vy == 1)).sum(); fp = ((p == 1) & (vy == 0)).sum(); fn = ((p == 0) & (vy == 1)).sum()
    rec = tp / max(tp + fn, 1); prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    if rec >= 0.90 and f1 > best_score:
        best_thr, best_score = float(thr), f1
print('Seuil conseille pour la webcam :', round(best_thr, 2))

cm = confusion_matrix(vy, (vp > best_thr).astype(int))
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=14)
plt.xticks([0, 1], ['No Fall', 'Fall']); plt.yticks([0, 1], ['No Fall', 'Fall'])
plt.xlabel('Predit'); plt.ylabel('Reel'); plt.title('Matrice de confusion (validation)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrix_v2.png'); plt.show()

CONFIG_PATH = f'{OUT_DIR}/config.json'
with open(CONFIG_PATH, 'w') as fh:
    json.dump(dict(T=T, FPS_ASSUMED=FPS_ASSUMED, WINDOW_SEC=WINDOW_SEC,
                   THRESHOLD=best_thr, DATA_Y_WAS_FLIPPED=Y_FLIP), fh, indent=2)
print('Ecrit :', MODEL_PATH, 'et', CONFIG_PATH)

if IN_COLAB:
    from google.colab import files
    files.download(MODEL_PATH); files.download(CONFIG_PATH)